In [16]:
import os
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_core.runnables import RunnableBranch,RunnableLambda
from langchain_core.output_parsers import PydanticOutputParser,StrOutputParser
from pydantic import BaseModel, Field
from typing import Literal

In [7]:
class sentement(BaseModel):
    sentiment: Literal["positive", "negative"] = Field(..., description="The sentiment of the text, either 'positive' or 'negative'.")

In [8]:
parser = PydanticOutputParser(pydantic_object=sentement)

In [9]:
promtp1 = PromptTemplate(template="Generate the sentiment of the following {feedback}in {format}", input_variables=["feedback"], partial_variables={"format": parser.get_format_instructions()})   

In [10]:
MODEL1 = "openai/gpt-oss-120b"
MODEL2 = "openai/gpt-oss-20b"
model = ChatGroq(
    model=MODEL1,
    temperature=0,
)
model1 = ChatGroq(
    model=MODEL2,
    temperature=0,
)

In [11]:
chain1 = promtp1 | model | parser

In [12]:
chain1.invoke({"I love this product! It's amazing."})

sentement(sentiment='positive')

In [13]:
prompt2 = PromptTemplate(template= "Generate the positive response for the {sentement} recived to the user", input_variables=["sentement"])

In [18]:
prompt3 = PromptTemplate(template= "Generate the  response as a sorry message for the {sentement} recived to the user", input_variables=["sentement"])

In [19]:
strparser = StrOutputParser()

In [24]:
conditionalChain = RunnableBranch((lambda x: x.sentiment == "positive",prompt2|model1|strparser),(lambda x: x.sentiment == "negative",prompt3|model1|strparser),RunnableLambda(lambda x:"Invalid Sentiment"))

In [25]:
final_response = chain1 | conditionalChain

In [27]:
final_response.invoke({"I didnt love  this Product ."})

'I’m really sorry to hear how disappointed you feel. Your experience matters to us, and I’d like to understand what went wrong so we can make things right. Please let me know how I can help.'

In [29]:
final_response.get_graph().print_ascii()

    +-------------+      
    | PromptInput |      
    +-------------+      
            *            
            *            
            *            
   +----------------+    
   | PromptTemplate |    
   +----------------+    
            *            
            *            
            *            
      +----------+       
      | ChatGroq |       
      +----------+       
            *            
            *            
            *            
+----------------------+ 
| PydanticOutputParser | 
+----------------------+ 
            *            
            *            
            *            
       +--------+        
       | Branch |        
       +--------+        
            *            
            *            
            *            
    +--------------+     
    | BranchOutput |     
    +--------------+     
